In [1]:
!pip install pytorch-lightning
!pip install pytorch-forecasting

   ---------------------------------------- 0.0/852.4 kB ? eta -:--:--
   --------------------------------------- 852.4/852.4 kB 12.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/983.4 kB ? eta -:--:--
   --------------------------------------- 983.4/983.4 kB 23.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/848.6 kB ? eta -:--:--
   --------------------------------------- 848.6/848.6 kB 18.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

import torch

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting import QuantileLoss

from pytorch_lightning import Trainer

In [4]:
data = pd.read_csv(
    r"C:\Users\pilli\Downloads\M5_Thesis_Project\data\processed\feature_engineered_data.csv"
)

data.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,snap_WI,sell_price,week,lag_7,lag_28,rolling_mean_7,rolling_mean_28,is_weekend,month_start,month_end
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1178,0,2014-04-20,11412,...,0,8.26,16,0.0,0.0,0.571429,0.392857,0,0,0
1,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1234,2,2014-06-15,11420,...,1,8.26,24,0.0,1.0,0.428571,0.428571,0,0,0
2,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_828,1,2013-05-05,11315,...,1,3.97,18,0.0,0.0,0.142857,0.142857,0,0,0
3,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1178,0,2014-04-20,11412,...,0,3.97,16,0.0,0.0,0.142857,0.142857,0,0,0
4,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1234,0,2014-06-15,11420,...,1,3.97,24,2.0,0.0,0.571429,0.500000,0,0,0


In [7]:
data['date'] = pd.to_datetime(data['date'])

In [9]:
data = data.sort_values('date')

data['time_idx'] = (
    data['date'] - data['date'].min()
).dt.days

data[['date','time_idx']].head()

,date,time_idx
862,2011-04-24,0
257,2011-04-24,0
1388,2011-04-24,0
1065,2011-04-24,0
251,2011-04-24,0


In [17]:
data['series_id'] = data['item_id']

In [13]:
training_cutoff = data["time_idx"].max() - 28

training = data[
    data.time_idx <= training_cutoff
]

validation = data[
    data.time_idx > training_cutoff
]

print(training.shape)
print(validation.shape)

(1123, 32)
(487, 32)


In [15]:
training.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,week,lag_7,lag_28,rolling_mean_7,rolling_mean_28,is_weekend,month_start,month_end,time_idx,series_id
862,HOBBIES_1_278_CA_1_validation,HOBBIES_1_278,HOBBIES_1,HOBBIES,CA_1,CA,d_86,28,2011-04-24,11113,...,16,10.0,33.0,7.714286,21.928571,0,0,0,0,HOBBIES_1_278
257,HOBBIES_1_082_CA_1_validation,HOBBIES_1_082,HOBBIES_1,HOBBIES,CA_1,CA,d_86,1,2011-04-24,11113,...,16,1.0,0.0,0.285714,0.357143,0,0,0,0,HOBBIES_1_082
1388,HOBBIES_2_011_CA_1_validation,HOBBIES_2_011,HOBBIES_2,HOBBIES,CA_1,CA,d_86,0,2011-04-24,11113,...,16,0.0,0.0,0.857143,0.321429,0,0,0,0,HOBBIES_2_011
1065,HOBBIES_1_338_CA_1_validation,HOBBIES_1_338,HOBBIES_1,HOBBIES,CA_1,CA,d_86,2,2011-04-24,11113,...,16,0.0,1.0,1.285714,1.428571,0,0,0,0,HOBBIES_1_338
251,HOBBIES_1_080_CA_1_validation,HOBBIES_1_080,HOBBIES_1,HOBBIES,CA_1,CA,d_86,3,2011-04-24,11113,...,16,0.0,0.0,2.428571,5.107143,0,0,0,0,HOBBIES_1_080


In [21]:
data.isnull().sum()

id                 0
item_id            0
dept_id            0
cat_id             0
store_id           0
state_id           0
d                  0
sales              0
date               0
wm_yr_wk           0
weekday            0
wday               0
month              0
year               0
event_name_1       0
event_type_1       0
event_name_2       0
event_type_2       0
snap_CA            0
snap_TX            0
snap_WI            0
sell_price         0
week               0
lag_7              0
lag_28             0
rolling_mean_7     0
rolling_mean_28    0
is_weekend         0
month_start        0
month_end          0
time_idx           0
series_id          0
dtype: int64

In [27]:
print("Unique Series:", data['series_id'].nunique())

print("Min Time:", data['time_idx'].min())

print("Max Time:", data['time_idx'].max())

Unique Series: 487
Min Time: 0
Max Time: 1148


In [37]:
print("Training Shape:", training.shape)

print("Unique Series:", training['series_id'].nunique())

print(
    training.groupby('series_id')
    .size()
    .describe()
)

Training Shape: (1123, 32)
Unique Series: 475
count    475.000000
mean       2.364211
std        0.711556
min        1.000000
25%        2.000000
50%        3.000000
75%        3.000000
max        3.000000
dtype: float64


In [41]:
training.groupby('series_id').size().describe()

count    475.000000
mean       2.364211
std        0.711556
min        1.000000
25%        2.000000
50%        3.000000
75%        3.000000
max        3.000000
dtype: float64

In [47]:

series_lengths = training.groupby("series_id")["time_idx"].count()
print(series_lengths.describe())
print(f"Series with < 88 steps: {(series_lengths < 88).sum()}")
print(f"Shortest series: {series_lengths.min()}")

count    475.000000
mean       2.364211
std        0.711556
min        1.000000
25%        2.000000
50%        3.000000
75%        3.000000
max        3.000000
Name: time_idx, dtype: float64
Series with < 88 steps: 475
Shortest series: 1


In [53]:
max_encoder_length = 30
min_encoder_length = max_encoder_length

In [55]:

min_required_length = max_encoder_length + max_prediction_length

valid_series = (
    training.groupby("series_id")["time_idx"]
    .count()
    .loc[lambda x: x >= min_required_length]
    .index
)

training_filtered = training[training["series_id"].isin(valid_series)]
print(f"Kept {len(valid_series)} series out of {training['series_id'].nunique()}")

Kept 0 series out of 475


In [57]:

for sid, df in training.groupby("series_id"):
    span = df["time_idx"].max() - df["time_idx"].min() + 1
    print(sid, span)
    break

HOBBIES_1_001 1


In [61]:

df = df.rename(columns={'sales': 'sales_old'}) 

training = df.melt(
    id_vars=['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    var_name='date',
    value_name='sales'
)
training['series_id'] = training['item_id'] + '_' + training['store_id']
training['time_idx'] = training.groupby('series_id').cumcount()

In [65]:

training = df.melt(id_vars=['item_id']) 

training = df.melt(
    id_vars=['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    var_name='date',
    value_name='sales'
)
training['series_id'] = training['item_id'] + '_' + training['store_id']
training['time_idx'] = training.groupby('series_id').cumcount()

In [67]:
print("Rows per series:")
print(training.groupby('series_id').size().describe())

Rows per series:
count     1.0
mean     27.0
std       NaN
min      27.0
25%      27.0
50%      27.0
75%      27.0
max      27.0
dtype: float64


In [77]:
import pandas as pd

data = pd.read_csv(
    r"C:\Users\pilli\Downloads\M5_Thesis_Project\data\processed\feature_engineered_data.csv"
)

print(data.shape)
print(data.columns.tolist())

(1610, 30)
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'week', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'is_weekend', 'month_start', 'month_end']


In [79]:
data.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,snap_WI,sell_price,week,lag_7,lag_28,rolling_mean_7,rolling_mean_28,is_weekend,month_start,month_end
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1178,0,2014-04-20,11412,...,0,8.26,16,0.0,0.0,0.571429,0.392857,0,0,0
1,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1234,2,2014-06-15,11420,...,1,8.26,24,0.0,1.0,0.428571,0.428571,0,0,0
2,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_828,1,2013-05-05,11315,...,1,3.97,18,0.0,0.0,0.142857,0.142857,0,0,0
3,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1178,0,2014-04-20,11412,...,0,3.97,16,0.0,0.0,0.142857,0.142857,0,0,0
4,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1234,0,2014-06-15,11420,...,1,3.97,24,2.0,0.0,0.571429,0.500000,0,0,0


In [81]:
data.shape

(1610, 30)

In [1]:
import pandas as pd

data = pd.read_csv(
    r"C:\Users\pilli\Downloads\M5_Thesis_Project\data\processed\feature_engineered_data.csv"
)

print("Dataset Shape:")
print(data.shape)

print("\nUnique IDs:")
print(data['id'].nunique())

print("\nRows per Series:")
print(
    data.groupby('id').size().sort_values().head(20)
)

Dataset Shape:
(1610, 30)

Unique IDs:
487

Rows per Series:
id
HOBBIES_2_024_CA_1_validation    1
HOBBIES_2_012_CA_1_validation    1
HOBBIES_2_025_CA_1_validation    1
HOBBIES_2_031_CA_1_validation    1
HOBBIES_2_038_CA_1_validation    1
HOBBIES_1_091_CA_1_validation    1
HOBBIES_2_035_CA_1_validation    1
HOBBIES_2_043_CA_1_validation    1
HOBBIES_2_058_CA_1_validation    1
HOBBIES_2_065_CA_1_validation    1
HOBBIES_2_082_CA_1_validation    1
HOBBIES_2_041_CA_1_validation    1
HOBBIES_1_183_CA_1_validation    2
HOBBIES_1_170_CA_1_validation    2
HOBBIES_1_171_CA_1_validation    2
HOBBIES_1_180_CA_1_validation    2
HOBBIES_1_190_CA_1_validation    2
HOBBIES_1_001_CA_1_validation    2
HOBBIES_1_203_CA_1_validation    2
HOBBIES_1_211_CA_1_validation    2
dtype: int64
